# F1 Qualifying Time Prediction

โปรเจกต์ Regression สำหรับทำนายเวลารอบ Qualifying โดยแยกขั้นตอนข้อมูลอย่างชัดเจน:

`FastF1 → Raw Tables → Data Preparation → ML Preprocessing → Models → Evaluation → Streamlit`

ข้อมูลครอบคลุมฤดูกาล 2021–2023 และใช้ปี 2023 เป็น test set

## 1. Setup

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

RAW_DIR = Path('data/raw')
PROCESSED_PATH = Path('data/processed/f1_model_features.csv')
pd.set_option('display.max_columns', None)
print('Setup ready')

## 2. Raw Data Extraction

ชั้นนี้เก็บข้อมูลตาม granularity ที่ FastF1 ส่งมาโดยไม่ aggregate, impute หรือ encode:

- `qualifying_results.csv`: หนึ่งแถวต่อนักแข่ง/สนาม พร้อม Q1, Q2, Q3
- `practice_laps.csv`: หนึ่งแถวต่อ lap พร้อม sector, speed, tyre และ quality flags
- `qualifying_weather.csv`: หนึ่งแถวต่อ weather timestamp

เวลา Timedelta ถูกแปลงเป็นวินาทีเพื่อบันทึก CSV เท่านั้น ไม่มีการเติมค่า

In [ ]:
from data_collection import collect_raw_dataset

# CSV ถูกเตรียมไว้แล้ว จึงไม่ดาวน์โหลดซ้ำเมื่อ Run All
# เปลี่ยนเป็น True เฉพาะเมื่อต้องการสร้าง raw data ใหม่จาก FastF1
RUN_RAW_EXTRACTION = False
if RUN_RAW_EXTRACTION:
    collect_raw_dataset(force=True)

In [ ]:
raw_results = pd.read_csv(RAW_DIR / 'qualifying_results.csv')
raw_laps = pd.read_csv(RAW_DIR / 'practice_laps.csv', low_memory=False)
raw_weather = pd.read_csv(RAW_DIR / 'qualifying_weather.csv')

raw_summary = pd.DataFrame({
    'Table': ['Qualifying results', 'Practice laps', 'Qualifying weather'],
    'Rows': [len(raw_results), len(raw_laps), len(raw_weather)],
    'Columns': [raw_results.shape[1], raw_laps.shape[1], raw_weather.shape[1]],
    'Duplicate rows': [raw_results.duplicated().sum(), raw_laps.duplicated().sum(), raw_weather.duplicated().sum()],
})
display(raw_summary)
display(raw_results.head())
display(raw_laps.head())
display(raw_weather.head())

## 3. Data Preparation / Feature Engineering

ขั้นนี้เปลี่ยน raw tables ให้เป็นหนึ่งแถวต่อนักแข่ง/สนาม:

1. ลบผล Qualifying ที่ซ้ำด้วย `Year + Circuit + Driver`
2. สร้าง `QualiTime = min(Q1, Q2, Q3)`
3. ตัด practice lap ที่ไม่มีเวลา/Driver, ถูกลบ หรือ `IsAccurate=False`
4. หาเวลาที่ดีที่สุดของ FP1, FP2 และ FP3 ต่อคน
5. เฉลี่ย weather timestamps ต่อสนาม
6. Merge ตารางทั้งหมด โดยยังเก็บ missing values ไว้ให้ ML preprocessing จัดการ

In [ ]:
from data_preparation import prepare_modeling_table

df_features, preparation_report = prepare_modeling_table(
    raw_dir=RAW_DIR,
    output_path=PROCESSED_PATH,
)
display(pd.Series(preparation_report, name='Value').to_frame())
display(df_features.head())

In [ ]:
stage_comparison = pd.DataFrame({
    'Stage': ['Raw practice laps', 'Usable practice laps', 'Prepared modeling rows'],
    'Rows': [
        preparation_report['raw_lap_rows'],
        preparation_report['usable_lap_rows'],
        preparation_report['modeling_rows'],
    ],
})
display(stage_comparison)

missing_prepared = df_features.isna().sum()
display(missing_prepared[missing_prepared > 0].rename('Missing values').to_frame())

## 4. ML Preprocessing

ขั้นนี้จึงค่อยจัดการข้อมูลสำหรับโมเดล: duplicate, invalid target, missing value, encoding, scaling, feature selection และ train/test split

เพื่อป้องกัน data leakage ทุก transformer เรียนรู้จาก train set เท่านั้น และ Target Encoding ของ train ใช้ out-of-fold encoding

In [ ]:
from preprocessing import prepare_data

prepared = prepare_data(df_features, test_year=2023, max_features=15)
X_train, X_test = prepared.X_train, prepared.X_test
y_train, y_test = prepared.y_train, prepared.y_test

print(prepared.split_description)
print(f'Raw prepared rows: {len(df_features):,}')
print(f'Invalid target rows removed: {prepared.report["invalid_target_rows_removed"]:,}')
print(f'Train shape: {X_train.shape}; Test shape: {X_test.shape}')
print(f'Remaining missing values: {prepared.report["remaining_missing_values"]}')
display(pd.Series(prepared.report['selected_features'], name='Selected feature'))

In [ ]:
assert len(X_train) == len(y_train)
assert len(X_test) == len(y_test)
assert list(X_train.columns) == list(X_test.columns)
assert not X_train.isna().any().any()
assert not X_test.isna().any().any()
assert set(X_train.index).isdisjoint(X_test.index)
print('Preprocessing validation passed')

## 5. Models and Evaluation

Train Linear Regression และ Random Forest ด้วยปี 2021–2022 แล้วประเมินบนปี 2023 ด้วย MAE, MSE, RMSE และ R²

In [ ]:
from modeling import train_models

model_bundle = train_models(
    dataset_path=PROCESSED_PATH,
    artifact_dir='artifacts',
)
evaluation = model_bundle['metrics']
display(evaluation.style.format({'MAE': '{:.3f}', 'MSE': '{:.3f}', 'RMSE': '{:.3f}', 'R2': '{:.3f}'}))

In [ ]:
ax = evaluation.plot.bar(x='Model', y='RMSE', legend=False, color=['#e10600', '#15151e'])
ax.set_title('Model comparison on held-out 2023 season')
ax.set_ylabel('RMSE (seconds; lower is better)')
ax.tick_params(axis='x', rotation=0)
plt.show()

## 6. Prediction Application

โมเดลและ preprocessing artifacts ถูกบันทึกใน `artifacts/model_bundle.joblib` และใช้โดย `app.py`

- JupyterLab: http://localhost:8888
- Streamlit: http://localhost:8501

```powershell
docker compose up -d
```